In [1]:
import pandas as pd

In [ ]:
def merge_wind_datasets(df_prod, df_coords, df_weather):
    """Combines production, coordinates, and weather datasets for forecasting.
    
    Args:
        df_prod (pd.DataFrame): Dataframe with production and capacity.
        df_coords (pd.DataFrame): Dataframe with site coordinates (latitude, longitude).
        df_weather (pd.DataFrame): Dataframe with weather forecasts.
        
    Returns:
        pd.DataFrame: A single merged dataframe.
    """
    # Ensure delivery_time is in datetime format for both
    df_prod['delivery_time'] = pd.to_datetime(df_prod['delivery_time'])
    df_weather['delivery_time'] = pd.to_datetime(df_weather['delivery_time'])
    
    # Merge production and weather on site and time
    # Use an inner join to keep only rows present in both datasets
    df_merged = pd.merge(
        df_prod,
        df_weather,
        on=['site_name', 'delivery_time'],
        how='inner'
    )
    
    # Merge with coordinates to add latitude and longitude
    df_final = pd.merge(
        df_merged,
        df_coords[['site_name', 'latitude', 'longitude']],
        on='site_name',
        how='left'
    )
    
    # Sort by site and time for consistent time-series analysis
    df_final = df_final.sort_values(['site_name', 'delivery_time']).reset_index(drop=True)
    
    return df_final


In [ ]:
df_1 = pd.read_parquet("../data/dataset_1.parquet")
df_2 = pd.read_parquet("../data/dataset_2.parquet")
df_3 = pd.read_parquet("../data/dataset_3.parquet")

df_merged = merge_wind_datasets(df_1, df_2, df_3)
df_merged.to_csv("../data/merged_dataset.csv", index=False)